# Lambda GPU Optimization Workshop
## Algoverse AI Research — Hands-on GPU Lab

**Budget**: $400 per team — every wasted minute costs real dollars!

This notebook is designed to run on **both CPU and GPU** machines:
- **Teams WITH Lambda access**: run on your instance for real GPU benchmarks
- **Teams WITHOUT Lambda access**: run locally on CPU, study the patterns, complete the quiz

---

## Step 0: Connect to Your Lambda Instance

If you have Lambda access, run these commands in your **local terminal** first:

```bash
# 1. Set up your SSH key (one-time)
#    Go to https://cloud.lambdalabs.com → SSH Keys
#    Download your private key, then:
chmod 600 ~/.ssh/lambda_key

# 2. SSH into your instance
ssh -i ~/.ssh/lambda_key ubuntu@<your-instance-ip>

# 3. Copy this notebook to Lambda
scp -i ~/.ssh/lambda_key lambda_gpu_optimization.ipynb ubuntu@<your-instance-ip>:~/

# 4. Start Jupyter on Lambda
jupyter notebook --no-browser --port=8888

# 5. In another terminal, create an SSH tunnel
ssh -i ~/.ssh/lambda_key -L 8888:localhost:8888 ubuntu@<your-instance-ip>
# Then open http://localhost:8888 in your browser
```

No Lambda access? Just run the notebook locally — everything works on CPU too.

## Step 1: Install Dependencies & Detect Hardware

In [ ]:
!pip install -q torch transformers datasets accelerate

In [ ]:
import torch
import time
import os
import subprocess
import platform

# Detect hardware
HAS_GPU = torch.cuda.is_available()
DEVICE = "cuda" if HAS_GPU else "cpu"

print(f"PyTorch version : {torch.__version__}")
print(f"Platform        : {platform.system()} {platform.machine()}")
print(f"CUDA available  : {HAS_GPU}")
print(f"Using device    : {DEVICE}")

if HAS_GPU:
    print(f"GPU name        : {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU memory      : {mem_gb:.1f} GB")
else:
    print("\nNo GPU detected — running in CPU mode.")
    print("You can still run all cells, study the optimizations, and do the quiz.")

In [ ]:
# Check GPU details (only works on machines with NVIDIA GPUs)
try:
    result = subprocess.run(
        ["nvidia-smi"],
        capture_output=True, text=True, timeout=10
    )
    if result.returncode == 0:
        print(result.stdout)
    else:
        print("nvidia-smi returned an error (GPU may not be available)")
except FileNotFoundError:
    print("nvidia-smi not found — no NVIDIA GPU on this machine.")
    print("This is expected if you're running locally without a GPU.")
except Exception as e:
    print(f"Could not run nvidia-smi: {e}")

---
## Step 2: Load a Model for Demos

We'll use `bert-base-uncased` (small enough to run on CPU) for all the optimization demos.

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

print("Downloading bert-base-uncased (first run only, ~440MB)...")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
print("Tokenizer loaded.")

# We'll load the model fresh in each demo cell so they're independent
print("Ready for optimization demos!")

---
## Optimization 1: Don't Waste GPU on Data Loading

The GPU sits idle while the CPU prepares the next batch. Fix: parallel data loading.

**Impact**: 2-5x speedup for data-heavy workloads.

In [ ]:
from torch.utils.data import DataLoader, Dataset


class TextClassificationDataset(Dataset):
    """Pre-tokenizes all texts once (Fix #4: no tokenizing inside loop)."""

    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(
            texts, truncation=True, padding="max_length",
            max_length=max_length, return_tensors="pt"
        )
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": self.labels[idx]
        }


# Create a small demo dataset
demo_texts = ["This movie is great"] * 100 + ["This movie is terrible"] * 100
demo_labels = [1] * 100 + [0] * 100
dataset = TextClassificationDataset(demo_texts, demo_labels, tokenizer)

# BAD: no workers, no pinned memory
bad_loader = DataLoader(dataset, batch_size=32)

# GOOD: parallel loading + pinned memory
# Note: num_workers>0 can cause issues in Jupyter notebooks on some systems.
# In a .py script on Lambda, always use num_workers=4.
good_loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,          # Use 4 in .py scripts on Lambda!
    pin_memory=HAS_GPU,     # Only useful with GPU
)

# Verify it works
sample_batch = next(iter(good_loader))
print(f"Batch keys    : {list(sample_batch.keys())}")
print(f"Batch size    : {sample_batch['input_ids'].shape[0]}")
print(f"Sequence len  : {sample_batch['input_ids'].shape[1]}")
print(f"\nDataLoader ready! In a .py script on Lambda, add:")
print(f"  num_workers=4, pin_memory=True, prefetch_factor=2")

## Optimization 2: Mixed Precision Training

A100/H100 GPUs have Tensor Cores optimized for FP16. Mixed precision:
- Halves memory usage (fit 2x larger batches)
- 2-3x faster training

**Impact**: 2-3x speedup + 2x memory savings on A100/H100.

In [ ]:
from torch.amp import autocast, GradScaler

# Load a fresh model for this demo
model_mp = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=2
).to(DEVICE)
optimizer_mp = torch.optim.AdamW(model_mp.parameters(), lr=2e-5)

# --- FP32 training (BAD on GPU — slow, uses 2x memory) ---
model_mp.train()
batch = {k: v.to(DEVICE) for k, v in next(iter(good_loader)).items()}

start = time.time()
for _ in range(5):
    outputs = model_mp(**batch)
    outputs.loss.backward()
    optimizer_mp.step()
    optimizer_mp.zero_grad()
fp32_time = time.time() - start
print(f"FP32 (5 steps): {fp32_time:.3f}s")

# --- Mixed Precision training (GOOD — faster, less memory) ---
scaler = GradScaler(enabled=HAS_GPU)

start = time.time()
for _ in range(5):
    with autocast(device_type=DEVICE, enabled=HAS_GPU):
        outputs = model_mp(**batch)
        loss = outputs.loss
    if HAS_GPU:
        scaler.scale(loss).backward()
        scaler.step(optimizer_mp)
        scaler.update()
    else:
        loss.backward()
        optimizer_mp.step()
    optimizer_mp.zero_grad()
mp_time = time.time() - start
print(f"Mixed Precision (5 steps): {mp_time:.3f}s")

if HAS_GPU:
    print(f"\nSpeedup: {fp32_time / mp_time:.2f}x")
else:
    print("\nOn CPU, mixed precision has minimal effect.")
    print("On an A100, expect 2-3x speedup!")

# Cleanup
del model_mp, optimizer_mp, scaler
if HAS_GPU:
    torch.cuda.empty_cache()

## Optimization 3: Efficient Inference Batching

Running prompts one at a time = GPU at 5%. Batching = GPU at 90%+.

**Impact**: 10-20x faster for benchmarks and evaluations.

In [ ]:
# Load model for inference demo
model_inf = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=2
).to(DEVICE)
model_inf.eval()

test_texts = ["This is a great movie", "Terrible film", "I loved it"] * 20  # 60 texts

# === BAD: one at a time ===
start = time.time()
bad_results = []
with torch.no_grad():
    for text in test_texts:
        inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
        output = model_inf(**inputs)
        pred = output.logits.argmax(dim=-1).item()
        bad_results.append(pred)
one_by_one_time = time.time() - start
print(f"One-by-one ({len(test_texts)} texts): {one_by_one_time:.3f}s")

# === GOOD: batched inference ===
start = time.time()
good_results = []
with torch.no_grad():
    for i in range(0, len(test_texts), 16):  # batch_size=16
        batch_texts = test_texts[i:i+16]
        inputs = tokenizer(
            batch_texts, return_tensors="pt",
            padding=True, truncation=True
        ).to(DEVICE)
        outputs = model_inf(**inputs)
        preds = outputs.logits.argmax(dim=-1).tolist()
        good_results.extend(preds)
batched_time = time.time() - start
print(f"Batched ({len(test_texts)} texts):    {batched_time:.3f}s")

speedup = one_by_one_time / batched_time if batched_time > 0 else float('inf')
print(f"\nSpeedup: {speedup:.1f}x")
print(f"Results match: {bad_results == good_results}")

# Cleanup
del model_inf
if HAS_GPU:
    torch.cuda.empty_cache()

## Optimization 4: GPU Monitoring & Checkpointing

Always know what your GPU is doing. Always save checkpoints — spot instances can die anytime.

In [ ]:
import threading


def gpu_monitor(interval=5, stop_event=None):
    """Background thread that prints GPU utilization."""
    while not (stop_event and stop_event.is_set()):
        try:
            result = subprocess.run(
                ["nvidia-smi",
                 "--query-gpu=utilization.gpu,memory.used,memory.total",
                 "--format=csv,noheader,nounits"],
                capture_output=True, text=True, timeout=5
            )
            if result.returncode == 0:
                parts = result.stdout.strip().split(", ")
                print(f"[GPU Monitor] Utilization: {parts[0]}% | "
                      f"Memory: {parts[1]}MiB / {parts[2]}MiB")
        except (FileNotFoundError, Exception):
            print("[GPU Monitor] nvidia-smi unavailable")
            break
        time.sleep(interval)


def save_checkpoint(model, optimizer, epoch, loss, path="checkpoints"):
    """Save model + optimizer state to resume later."""
    os.makedirs(path, exist_ok=True)
    filepath = os.path.join(path, f"ckpt_epoch_{epoch}.pt")
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loss": loss
    }, filepath)
    print(f"  Checkpoint saved: epoch {epoch}, loss {loss:.4f}")
    return filepath


def load_checkpoint(model, optimizer, path):
    """Resume training from a checkpoint."""
    ckpt = torch.load(path, weights_only=False, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    print(f"  Resumed from epoch {ckpt['epoch']}, loss {ckpt['loss']:.4f}")
    return ckpt["epoch"]


# Test the monitor (runs for ~6 seconds)
print("Testing GPU monitor (will print 1-2 readings)...")
stop = threading.Event()
t = threading.Thread(target=gpu_monitor, args=(3, stop), daemon=True)
t.start()
time.sleep(6)
stop.set()
t.join(timeout=5)
print("GPU monitor test done.")

In [ ]:
# Test checkpointing
print("Testing checkpoint save/load...")

test_model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=2
).to(DEVICE)
test_optimizer = torch.optim.AdamW(test_model.parameters(), lr=2e-5)

# Save
ckpt_path = save_checkpoint(test_model, test_optimizer, epoch=0, loss=0.693)

# Load into a fresh model
test_model_2 = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=2
).to(DEVICE)
test_optimizer_2 = torch.optim.AdamW(test_model_2.parameters(), lr=2e-5)
load_checkpoint(test_model_2, test_optimizer_2, ckpt_path)

# Verify weights match
w1 = list(test_model.parameters())[0].data
w2 = list(test_model_2.parameters())[0].data
print(f"  Weights match after reload: {torch.equal(w1, w2)}")

# Cleanup
del test_model, test_model_2, test_optimizer, test_optimizer_2
if HAS_GPU:
    torch.cuda.empty_cache()

# Remove test checkpoint
import shutil
if os.path.exists("checkpoints"):
    shutil.rmtree("checkpoints")
    print("  Cleaned up test checkpoints.")

## Lambda Cost Cheat Sheet

| GPU | $/hr (on-demand) | $/hr (spot) | Best For |
|-----|------------------|-------------|----------|
| A10 (24GB) | ~$0.75 | ~$0.30 | Small model inference, fine-tuning ≤7B |
| A100 (80GB) | ~$1.50 | ~$0.60 | Training 7B-13B, large batch inference |
| H100 (80GB) | ~$2.50 | ~$1.00 | Training 13B+, fastest throughput |

**With a $400 budget:**
- A10 spot: ~1,333 hours (~55 days)
- A100 spot: ~667 hours (~27 days)  
- H100 spot: ~400 hours (~16 days)
- A100 on-demand: ~267 hours (~11 days)

**Budget Rules:**
1. Develop locally or on CPU first. Only move to GPU when code is tested.
2. Use spot instances for everything except time-critical deadlines.
3. `nvidia-smi` check every time — if util < 50%, stop and optimize first.
4. Set a Lambda budget alert so you don't get surprise bills.

---
## Full Exercise: The Slow Script vs The Optimized Script

Below we run the **slow way** and the **optimized way** side by side and compare timing.

We use a small sample (20 examples) so it finishes quickly even on CPU.

In [ ]:
# ============================================================
# THE SLOW WAY — 7 performance problems
# ============================================================
# Problems:
#   1. Batch size = 1 (one sample at a time)
#   2. No parallel data loading
#   3. Full FP32 (no mixed precision)
#   4. Tokenizing inside the loop
#   5. No gradient accumulation
#   6. No checkpointing
#   7. No monitoring

NUM_SAMPLES = 20  # Small for demo — real training uses 1000s

slow_model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=2
).to(DEVICE)
slow_optimizer = torch.optim.Adam(slow_model.parameters(), lr=2e-5)

texts = ["This movie is great"] * (NUM_SAMPLES // 2) + \
        ["This movie is terrible"] * (NUM_SAMPLES // 2)
labels = [1] * (NUM_SAMPLES // 2) + [0] * (NUM_SAMPLES // 2)

slow_model.train()
print(f"Running SLOW training on {NUM_SAMPLES} samples (batch_size=1)...")
start = time.time()

for i in range(len(texts)):
    inputs = tokenizer(texts[i], return_tensors="pt", padding=True,
                       truncation=True).to(DEVICE)
    outputs = slow_model(
        **inputs,
        labels=torch.tensor([labels[i]]).to(DEVICE)
    )
    outputs.loss.backward()
    slow_optimizer.step()
    slow_optimizer.zero_grad()

slow_time = time.time() - start
print(f"SLOW training done: {slow_time:.2f}s")
print(f"  Per sample: {slow_time / NUM_SAMPLES * 1000:.1f}ms")
print(f"  Projected for 1000 samples: {slow_time / NUM_SAMPLES * 1000:.1f}s")

# Cleanup
del slow_model, slow_optimizer
if HAS_GPU:
    torch.cuda.empty_cache()

In [ ]:
# ============================================================
# THE OPTIMIZED WAY — All 7 problems fixed
# ============================================================

BATCH_SIZE = 10  # Fix #1: batch instead of 1-by-1
ACCUMULATION_STEPS = 2  # Fix #5: effective batch = 10 * 2 = 20
CHECKPOINT_DIR = "demo_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Fix #4: Pre-tokenize into a Dataset
fast_dataset = TextClassificationDataset(texts, labels, tokenizer)

# Fix #1 & #2: Batched DataLoader
fast_loader = DataLoader(
    fast_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,         # Use 4 in .py scripts on Lambda
    pin_memory=HAS_GPU,
)

fast_model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=2
).to(DEVICE)
fast_optimizer = torch.optim.AdamW(fast_model.parameters(), lr=2e-5, weight_decay=0.01)

# Fix #3: Mixed Precision
scaler = GradScaler(enabled=HAS_GPU)

# Fix #7: GPU Monitoring
stop_monitor = threading.Event()
if HAS_GPU:
    threading.Thread(
        target=gpu_monitor, args=(5, stop_monitor), daemon=True
    ).start()

fast_model.train()
print(f"Running OPTIMIZED training on {NUM_SAMPLES} samples "
      f"(batch_size={BATCH_SIZE}, accum={ACCUMULATION_STEPS})...")
start = time.time()

for step, batch in enumerate(fast_loader):
    batch = {k: v.to(DEVICE) for k, v in batch.items()}

    # Fix #3: Mixed precision forward pass
    with autocast(device_type=DEVICE, enabled=HAS_GPU):
        outputs = fast_model(**batch)
        loss = outputs.loss / ACCUMULATION_STEPS  # Fix #5

    if HAS_GPU:
        scaler.scale(loss).backward()
    else:
        loss.backward()

    # Fix #5: Gradient accumulation — step every N batches
    if (step + 1) % ACCUMULATION_STEPS == 0:
        if HAS_GPU:
            scaler.step(fast_optimizer)
            scaler.update()
        else:
            fast_optimizer.step()
        fast_optimizer.zero_grad()

fast_time = time.time() - start

# Fix #6: Checkpointing
save_checkpoint(fast_model, fast_optimizer, epoch=0, loss=loss.item(),
                path=CHECKPOINT_DIR)

stop_monitor.set()  # Stop GPU monitor

print(f"\nOPTIMIZED training done: {fast_time:.2f}s")
print(f"  Per sample: {fast_time / NUM_SAMPLES * 1000:.1f}ms")
print(f"  Projected for 1000 samples: {fast_time / NUM_SAMPLES * 1000:.1f}s")

# Cleanup
del fast_model, fast_optimizer, scaler
if HAS_GPU:
    torch.cuda.empty_cache()
if os.path.exists(CHECKPOINT_DIR):
    shutil.rmtree(CHECKPOINT_DIR)

In [ ]:
# ============================================================
# COMPARISON
# ============================================================
print("=" * 50)
print("TIMING COMPARISON")
print("=" * 50)
print(f"Slow  (batch=1, FP32, no DataLoader) : {slow_time:.2f}s")
print(f"Fast  (batch={BATCH_SIZE}, mixed prec, DataLoader): {fast_time:.2f}s")
speedup = slow_time / fast_time if fast_time > 0 else float('inf')
print(f"Speedup                              : {speedup:.1f}x")
print()
if not HAS_GPU:
    print("Note: On CPU the speedup is modest (mainly from batching).")
    print("On an A100 GPU, expect 10-30x speedup from all optimizations combined!")
else:
    print("This is your real GPU speedup. Imagine this over hours of training!")
    saved_per_hour = (1 - 1/speedup) * 100
    print(f"You'd save ~{saved_per_hour:.0f}% of your Lambda bill.")

---
## Useful Lambda Commands

These cells demonstrate common operations on Lambda instances.

In [ ]:
# System info — works on both Linux (Lambda) and macOS (local)
print("=" * 40)
print("SYSTEM INFO")
print("=" * 40)

import multiprocessing
print(f"OS           : {platform.system()} {platform.release()}")
print(f"Architecture : {platform.machine()}")
print(f"CPU cores    : {multiprocessing.cpu_count()}")
print(f"Python       : {platform.python_version()}")
print(f"PyTorch      : {torch.__version__}")
print(f"CUDA avail   : {HAS_GPU}")
if HAS_GPU:
    print(f"CUDA version : {torch.version.cuda}")
    print(f"GPU          : {torch.cuda.get_device_name(0)}")

In [ ]:
# GPU utilization snapshot
try:
    result = subprocess.run(
        ["nvidia-smi",
         "--query-gpu=name,utilization.gpu,utilization.memory,memory.used,memory.total,temperature.gpu",
         "--format=csv"],
        capture_output=True, text=True, timeout=10
    )
    if result.returncode == 0:
        print("GPU Status:")
        print(result.stdout)
    else:
        print("nvidia-smi error:", result.stderr)
except FileNotFoundError:
    print("No NVIDIA GPU detected.")
    print("On Lambda, this would show something like:")
    print("  name, utilization.gpu [%], utilization.memory [%], memory.used [MiB], memory.total [MiB], temperature.gpu")
    print("  NVIDIA A100-SXM4-80GB, 85 %, 42 %, 34567 MiB, 81920 MiB, 62")

In [ ]:
# Disk space check
import shutil

total, used, free = shutil.disk_usage("/")
print(f"Disk total : {total / (1024**3):.1f} GB")
print(f"Disk used  : {used / (1024**3):.1f} GB")
print(f"Disk free  : {free / (1024**3):.1f} GB")
print(f"Usage      : {used/total*100:.1f}%")

if free / (1024**3) < 10:
    print("\nWARNING: Low disk space! Clear old checkpoints or model caches.")

### File Transfer Commands (run in local terminal)

```bash
# Copy a file TO Lambda
scp -i ~/.ssh/lambda_key my_script.py ubuntu@<ip>:~/

# Copy a file FROM Lambda
scp -i ~/.ssh/lambda_key ubuntu@<ip>:~/results.json ./

# Copy an entire folder FROM Lambda
scp -i ~/.ssh/lambda_key -r ubuntu@<ip>:~/checkpoints ./

# Run training in background (survives SSH disconnect)
# Option 1: tmux (recommended)
tmux new -s training
python train.py
# Press Ctrl+B, then D to detach
# tmux attach -t training  # to reconnect

# Option 2: nohup
nohup python train.py > train.log 2>&1 &
tail -f train.log  # watch progress
```

---
---
# Quiz: Lambda GPU Optimization

**Everyone should complete this quiz** — whether or not you have Lambda access.

This tests your understanding of GPU optimization concepts that are critical for not wasting your team's $400 budget.

Fill in your answers in the cells below, then run the answer key cell at the bottom to check.

### Question 1: Spot the Bug

What's wrong with this inference code? (There are 2 problems)

```python
model.train()
results = []
for text in test_texts:
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    output = model(**inputs)
    results.append(output.logits.cpu().numpy())
```

In [ ]:
q1_answer = """
Problem 1: YOUR ANSWER HERE
Problem 2: YOUR ANSWER HERE
"""
print("Your answer for Q1:")
print(q1_answer)

### Question 2: Cost Calculation

Your team has $400 in Lambda credits. You're fine-tuning a 7B model on an A100 (spot price: $0.60/hr).

- Each epoch takes 2 hours
- You need 5 epochs
- You want to run 3 hyperparameter configurations

**a)** How much will this cost?  
**b)** How much budget remains?  
**c)** If you switch to mixed precision, epochs take 45 minutes instead. New total cost?

In [ ]:
# Fill in the numbers
q2a = 0  # Total cost in dollars
q2b = 0  # Remaining budget in dollars
q2c = 0  # Mixed precision total cost in dollars

print(f"Your answers for Q2:")
print(f"  a) Total cost:           ${q2a:.2f}")
print(f"  b) Remaining budget:     ${q2b:.2f}")
print(f"  c) Mixed precision cost: ${q2c:.2f}")

### Question 3: Multiple Choice — DataLoader

Which DataLoader configuration is BEST for training on a Lambda A100?

```python
# A
DataLoader(dataset, batch_size=32)

# B
DataLoader(dataset, batch_size=32, num_workers=4)

# C
DataLoader(dataset, batch_size=32, num_workers=4, pin_memory=True)

# D
DataLoader(dataset, batch_size=32, num_workers=16, pin_memory=True, prefetch_factor=4)
```

In [ ]:
q3_answer = ""  # Type A, B, C, or D
print(f"Your answer for Q3: {q3_answer}")

### Question 4: True or False

Mark each statement as True or False:

1. `model.eval()` only affects layers like dropout and batchnorm, not speed.
2. `torch.no_grad()` reduces memory usage during inference.
3. Mixed precision training always gives identical results to FP32.
4. Checkpointing slows down training significantly.
5. `pin_memory=True` helps when your data is already on the GPU.
6. A spot instance can be terminated by Lambda at any time.

In [ ]:
# Fill in True or False for each
q4_answers = {
    1: "",  # model.eval() only affects dropout/batchnorm, not speed
    2: "",  # torch.no_grad() reduces memory during inference
    3: "",  # Mixed precision always gives identical results to FP32
    4: "",  # Checkpointing slows down training significantly
    5: "",  # pin_memory=True helps when data is already on GPU
    6: "",  # Spot instances can be terminated at any time
}

print("Your answers for Q4:")
for k, v in q4_answers.items():
    print(f"  {k}. {v if v else '(not answered)'}")

### Question 5: Debugging GPU Utilization

You run `nvidia-smi` during training and see:

```
| NVIDIA A100-SXM4-80GB | 12% | 4096MiB / 81920MiB |
```

GPU utilization is only 12% and memory usage is 4GB out of 80GB.

**List 3 likely causes and how to fix each one.**

In [ ]:
q5_answer = """
Cause 1: YOUR ANSWER
Fix:     YOUR ANSWER

Cause 2: YOUR ANSWER
Fix:     YOUR ANSWER

Cause 3: YOUR ANSWER
Fix:     YOUR ANSWER
"""
print("Your answer for Q5:")
print(q5_answer)

### Question 6: Code Fix

This gradient accumulation code has a bug. Find it and explain the fix.

```python
ACCUM_STEPS = 4
for step, batch in enumerate(train_loader):
    outputs = model(**batch)
    loss = outputs.loss
    loss.backward()
    
    if (step + 1) % ACCUM_STEPS == 0:
        optimizer.step()
        optimizer.zero_grad()
```

In [ ]:
q6_answer = """
Bug:  YOUR ANSWER
Fix:  YOUR ANSWER
Why:  YOUR ANSWER
"""
print("Your answer for Q6:")
print(q6_answer)

### Question 7: Budget Planning

Your team needs to:
1. Fine-tune a 7B model (needs A100, ~10 hours with mixed precision)
2. Run inference on 10,000 prompts (needs A10, ~3 hours with batching)
3. Train a small classifier from scratch (needs A10, ~5 hours)

Using spot prices (A10: $0.30/hr, A100: $0.60/hr):

**a)** What's the minimum cost for all three tasks?  
**b)** How many full re-runs can you afford with the remaining budget?  
**c)** What's ONE thing you'd do to reduce costs further?

In [ ]:
# Fill in the numbers
q7a = 0   # Minimum cost in dollars
q7b = 0   # Number of full re-runs affordable
q7c = ""  # One cost reduction idea

print(f"Your answers for Q7:")
print(f"  a) Minimum cost:     ${q7a:.2f}")
print(f"  b) Re-runs possible: {q7b}")
print(f"  c) Cost reduction:   {q7c if q7c else '(not answered)'}")

### Question 8: Ordering Optimizations

You just wrote a training script and it's slow. Rank these optimizations from **most impactful** (1) to **least impactful** (5) for a typical training job:

- A. Add gradient accumulation (effective batch size 32 -> 128)
- B. Switch from batch_size=1 to batch_size=32
- C. Add `pin_memory=True` to DataLoader
- D. Enable mixed precision (FP16)
- E. Set `num_workers=4` in DataLoader

In [ ]:
# Fill in the letters (A-E) in order of impact
q8_ranking = {
    1: "",  # Most impactful
    2: "",
    3: "",
    4: "",
    5: "",  # Least impactful
}

print("Your ranking for Q8 (most -> least impactful):")
for rank, letter in q8_ranking.items():
    print(f"  {rank}. {letter if letter else '(not answered)'}")

---
## Answer Key

**Run the cell below AFTER you've filled in ALL your answers above.**

In [ ]:
score = 0
total = 8

print("=" * 60)
print("ANSWER KEY — Lambda GPU Optimization Quiz")
print("=" * 60)

# --- Q1 ---
print("""
Q1: Spot the Bug (2 problems)
─────────────────────────────
Problem 1: model.train() should be model.eval()
  → Train mode keeps dropout active, giving inconsistent
    results. Also ~30% slower (batchnorm tracking stats).

Problem 2: Missing torch.no_grad()
  → Without it, PyTorch stores computation graphs for
    backprop, wasting memory. Wrap in:
    with torch.no_grad():

(Bonus: single-sample loop instead of batching.)
""")

# --- Q2 ---
correct_a = 3 * 5 * 2 * 0.60  # 18.00
correct_b = 400 - correct_a    # 382.00
correct_c = 3 * 5 * 0.75 * 0.60  # 6.75

q2_correct = (abs(q2a - correct_a) < 0.01 and
              abs(q2b - correct_b) < 0.01 and
              abs(q2c - correct_c) < 0.01)
if q2_correct:
    score += 1

print(f"""
Q2: Cost Calculation {'✓' if q2_correct else '✗'}
─────────────────────
a) 3 configs × 5 epochs × 2 hrs × $0.60/hr = ${correct_a:.2f}
   Your answer: ${q2a:.2f}
b) $400 - ${correct_a:.2f} = ${correct_b:.2f} remaining
   Your answer: ${q2b:.2f}
c) 3 configs × 5 epochs × 0.75 hrs × $0.60/hr = ${correct_c:.2f}
   Your answer: ${q2c:.2f}
   Savings: ${correct_a:.2f} - ${correct_c:.2f} = ${correct_a - correct_c:.2f} (62.5% reduction!)
""")

# --- Q3 ---
q3_correct = q3_answer.strip().upper() == "C"
if q3_correct:
    score += 1

print(f"""
Q3: DataLoader {'✓' if q3_correct else '✗'}
──────────────────────────────────
Answer: C   (Your answer: {q3_answer.strip().upper() if q3_answer.strip() else 'none'})

A: No workers, no pinned memory — GPU waits for CPU.
B: Workers help, but no pin_memory slows transfers.
C: Best practical config — 4 workers + pinned memory.
D: 16 workers is overkill — too many processes competing
   for CPU, diminishing returns. (C or D acceptable)
""")

# --- Q4 ---
q4_correct_answers = {1: "False", 2: "True", 3: "False",
                      4: "False", 5: "False", 6: "True"}
q4_score = sum(1 for k in q4_correct_answers
               if q4_answers.get(k, "").strip().lower() ==
                  q4_correct_answers[k].lower())
q4_passed = q4_score >= 5
if q4_passed:
    score += 1

print(f"""
Q4: True or False ({q4_score}/6) {'✓' if q4_passed else '✗'}
───────────────────
1. FALSE — model.eval() also speeds up inference by ~30%.
   Your answer: {q4_answers.get(1, '(none)')}
2. TRUE  — no_grad() prevents storing computation graphs.
   Your answer: {q4_answers.get(2, '(none)')}
3. FALSE — mixed precision can cause tiny numerical differences.
   Your answer: {q4_answers.get(3, '(none)')}
4. FALSE — checkpoint save takes < 1 second per epoch.
   Your answer: {q4_answers.get(4, '(none)')}
5. FALSE — pin_memory helps CPU→GPU transfers only.
   Your answer: {q4_answers.get(5, '(none)')}
6. TRUE  — spot instances are preemptible (why we checkpoint!).
   Your answer: {q4_answers.get(6, '(none)')}
""")

# --- Q5 ---
print("""
Q5: Debugging GPU Utilization (12%, 4GB/80GB)
──────────────────────────────────────────────
Likely causes (any 3 of these count):

1. Batch size too small
   Fix: increase batch_size (76GB free — use it!)
2. Data loading bottleneck (num_workers=0)
   Fix: add num_workers=4, pin_memory=True
3. CPU preprocessing inside the training loop
   Fix: pre-tokenize or move to Dataset.__getitem__
4. Calling .item()/.cpu() every step (GPU→CPU sync)
   Fix: only log every N steps
5. Model too small for this GPU
   Fix: use a cheaper GPU (A10) or increase batch size
""")

# --- Q6 ---
print("""
Q6: Gradient Accumulation Bug
──────────────────────────────────────────
Bug: loss is NOT divided by ACCUM_STEPS before .backward()

Without division, after 4 accumulated backward passes the
gradients are 4x too large — equivalent to a 4x higher
learning rate, causing unstable training.

Fix: Change line to:
  loss = outputs.loss / ACCUM_STEPS
""")

# --- Q7 ---
correct_7a = 10 * 0.60 + 3 * 0.30 + 5 * 0.30  # 8.40
correct_7b = int((400 - correct_7a) / correct_7a)  # 46

q7a_correct = abs(q7a - correct_7a) < 0.10
if q7a_correct:
    score += 1

print(f"""
Q7: Budget Planning {'✓' if q7a_correct else '✗'}
─────────────────────
a) Fine-tune: 10 hrs × $0.60 = $6.00
   Inference: 3 hrs × $0.30  = $0.90
   Classifier: 5 hrs × $0.30 = $1.50
   TOTAL: ${correct_7a:.2f}
   Your answer: ${q7a:.2f}

b) Budget left: $400 - ${correct_7a:.2f} = ${400 - correct_7a:.2f}
   Re-runs: ${400 - correct_7a:.2f} / ${correct_7a:.2f} ≈ {correct_7b} full re-runs
   Your answer: {q7b}

c) Valid cost reduction ideas:
   - Test code on CPU first before using GPU time
   - Use mixed precision to cut fine-tuning hours
   - Batch inference more aggressively
   - Use a smaller model if quality is acceptable
   Your answer: {q7c if q7c else '(none)'}
""")

# --- Q8 ---
correct_order = ["B", "D", "E", "A", "C"]
user_order = [q8_ranking.get(i, "").strip().upper() for i in range(1, 6)]
q8_correct = user_order == correct_order
if q8_correct:
    score += 1

print(f"""
Q8: Optimization Ranking {'✓' if q8_correct else '✗'}
─────────────────────────────────────────────────────
Correct ranking (most → least impactful):
  1. B — Batch size 1→32 (10-30x speedup — biggest win)
  2. D — Mixed precision (2-3x faster, 2x memory savings)
  3. E — num_workers=4 (2-5x faster data loading)
  4. A — Gradient accumulation (memory savings, not speed)
  5. C — pin_memory (modest 10-20% transfer speedup)

Your ranking: {' → '.join(user_order) if any(user_order) else '(not answered)'}
""")

# --- Final score ---
# Q1, Q5, Q6 are free-text — add points manually
print("=" * 60)
print(f"AUTO-SCORED: {score}/{total} (Q1, Q5, Q6 need manual review)")
print("Add 1 point each for Q1, Q5, Q6 if your answers were correct.")
print("=" * 60)
print()
print("Scoring Guide:")
print("  7-8: GPU Wizard — ready for Lambda")
print("  5-6: Solid — review what you missed")
print("  3-4: Getting There — re-read the optimization sections")
print("  0-2: Study More — pair with a teammate for your first Lambda session")
print()
print("Remember: every dollar wasted on bad GPU utilization")
print("is a dollar NOT spent on actual research.")